<a href="https://colab.research.google.com/github/annaphuongwit/14_LLMs/blob/main/2_chatbot_KE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Building a Conversational Chatbot

This guide provides instructions for extending your Large Language Model (LLM) application into a conversational chatbot. You should have already completed the setup in the first guide and have a working single-response LLM application.

> This notebook is a guide to enhancing your project. **Do not run the code cells here directly.** Follow the instructions in your local development environment.

-----

## 1.&nbsp; Library Installation

First, we'll install the necessary Python library for managing chat interactions. This command should be executed in your VSCode terminal with your Conda environment activated.

In the VSCode terminal, run the following command to install the `llama-index-core` library.

In [ ]:
# conda install conda-forge::llama-index-core

-----

## 2.&nbsp; From a Single Answer to a Real Conversation

Before we write any code, it's important to understand a key conceptual leap we're about to make.

In the first part of our project, our application was stateless. We used the `llm.complete()` method, which took a single question and returned a single answer. It had no memory of past interactions. If you asked it a follow-up question, it would have no idea what you were talking about.

To build a true chatbot, we need a stateful system - one that can remember the conversation history. This is the primary job of a Chat Engine. A chat engine wraps the LLM and automatically manages the back-and-forth dialogue, giving the model the context it needs to hold a coherent conversation. In this guide, we'll start with LlamaIndex's `SimpleChatEngine`.

-----

## 3.&nbsp; Update the Configuration with a System Prompt

A chatbot needs instructions on how to behave. We'll add a system prompt to our configuration file to define its personality.

1.  Open `src/config.py` in your VSCode editor.
2.  Comment out `LLM_QUESTION`, we no longer need it.
2.  Add the `LLM_SYSTEM_PROMPT` variable to the file. Your `config.py` should now look like this:

In [ ]:
from config_draft import *

In [ ]:
# # --- LLM Model Configuration ---
# LLM_MODEL: str = "llama-3.1-8b-instant"
# LLM_MAX_NEW_TOKENS: int = 768
# LLM_TEMPERATURE: float = 0.01
# LLM_TOP_P: float = 0.95
# LLM_REPETITION_PENALTY: float = 1.03
# # LLM_QUESTION: str = "What is the capital of France?" # Example question, not used in the current code
# LLM_SYSTEM_PROMPT: str = "You are a helpful chatbot. Be friendly and conversational."a

You might be thinking, "Couldn't I just tell the user to start their chat with the instructions for the personality they would like?" While you could, a system prompt is fundamentally different and more powerful for two reasons:

1.  It's a Meta-Instruction: A system prompt operates at a level above the user's conversation. It sets the "rules of the game" for the AI, which the model is heavily incentivised to follow throughout the entire dialogue.
2.  It's Persistent: The chat engine ensures the system prompt is always present. A user's instruction in the first turn might get "forgotten" or lose influence as the conversation history grows, but the system prompt is a constant reminder of the AI's core persona and directives.

-----

## 4.&nbsp; Creating an Interactive Chat Loop

Now we'll modify our engine to support a back-and-forth conversation. To properly understand how chat applications work, we'll first build the interactive loop manually using basic Python.

### 4.1 The Manual Method: Building a `while` Loop

1.  Open `src/engine/engine.py` in the VSCode editor.
2.  Replace the file's content with the code below. This script creates a `SimpleChatEngine` and a `while` loop that continuously prompts you for input.

In [ ]:
import os
from llama_index.llms.groq import Groq

def initialise_llm() -> Groq:
    """Initialises the Groq LLM with core parameters from config."""

    api_key : str | None = os.getenv("GROQ_API_KEY")

    if not api_key:
        raise ValueError("GROQ_API_KEY not found. Make sure it's set in your .env file.")

    return Groq(
        api_key=api_key,
        model=LLM_MODEL,
        # The following parameters are optional and will default to the model's defaults if not set
        # max_new_tokens=LLM_MAX_NEW_TOKENS,
        # temperature=LLM_TEMPERATURE,
        # top_p=LLM_TOP_P,
        # repetition_penalty=LLM_REPETITION_PENALTY,
    )

In [ ]:
from llama_index.core.chat_engine import SimpleChatEngine # adds a memory by default
from llama_index.llms.groq import Groq

# from src.config import LLM_SYSTEM_PROMPT
# from src.engine.components import initialise_llm

def main_chat_loop_w_memory() -> None:
    """Main chat loop for a conversational chatbot."""
    llm: Groq = initialise_llm()

    # Create a chat engine using the system prompt
    conversation = SimpleChatEngine.from_defaults(
        llm=llm,
        system_prompt=LLM_SYSTEM_PROMPT
    )

    print("--- Chat Started ---")
    print("Type 'exit' or 'quit' to end the conversation.")
    print("--------------------")

    while True:
        user_input: str = input("\nYou 😎: ").strip()
        # .strip() removes leading and trailing whitespace (spaces, tabs, newlines) from the input string.

        if user_input.lower() in ["quit", "exit"]:
            print("\nBot: Goodbye!")
            break

        if not user_input:
            continue

        response = conversation.chat(user_input) # vs Groq.complete()
        print(f"\nBot 🫡: {response}")

In [ ]:
main_chat_loop_w_memory()

--- Chat Started ---
Type 'exit' or 'quit' to end the conversation.
--------------------

Bot: Unfortunately, it's quite challenging to give an exact number of dogs on the planet, as it's constantly changing due to various factors like births, deaths, and human migration. However, I can provide some interesting statistics that might give you an idea.

According to the World Canine Organization (Fédération Cynologique Internationale), there are approximately 987 breeds of dogs recognized worldwide. But let's focus on the number of dogs.

Estimates suggest that there are around 1 billion dogs on the planet. Yes, you read that right - 1 billion! This number is based on various studies, surveys, and data from animal welfare organizations.

To break it down further, here are some approximate numbers:

- In the United States alone, there are around 78 million dogs.
- In Europe, there are approximately 100 million dogs.
- In Asia, the number is estimated to be around 150 million dogs.

Keep i

You might wonder how the `SimpleChatEngine` "remembers" the conversation without us writing any code to store the history.

Behind the scenes, the engine is maintaining a simple list of messages. Every time you call `conversation.chat()`, it does the following:

1.  It takes your new message.
2.  It appends it to the list of all previous user and AI messages.
3.  It sends the entire list to the LLM API.
4.  It gets the AI's response back and adds that to the list as well.

This is why the conversation feels natural. However, it also reveals a critical limitation: with every turn, the amount of text sent to the API grows. This can eventually become slow and exceed the model's context window. We'll explore a solution to this in the challenges section.

With the interactive loop in place, run the application from the VSCode terminal to start a conversation with your bot.

In [ ]:
# python main.py

### 4.2 The Easy Way: Using `chat_repl()`

Now that you understand the mechanics of building a chat loop from scratch, let's use the convenient, built-in method that LlamaIndex provides. This function, `chat_repl()`, handles the entire interactive loop for you.

1.  Open `src/engine/engine.py` in the VSCode editor again.
2.  Replace the file's content with the code below. Notice that the entire `while` loop has been replaced by a single line.

In [ ]:
from llama_index.core.chat_engine import SimpleChatEngine
# from src.config import LLM_SYSTEM_PROMPT
# from src.engine.components import initialise_llm

def main_chat_loop_w_memory_repl() -> None:
    """Main chat loop for a conversational chatbot."""
    llm: Groq = initialise_llm()

    conversation = SimpleChatEngine.from_defaults(
        llm=llm,
        system_prompt=LLM_SYSTEM_PROMPT
    )

    conversation.chat_repl()

In [ ]:
main_chat_loop_w_memory_repl()

===== Entering Chat REPL =====
Type "exit" to exit.

Assistant: As of my knowledge cutoff in 2023, the President of the United States was Joe Biden. However, please note that my information may not be up to date, and I recommend checking a reliable news source for the most current information.

If you're looking for the latest information, I can suggest some reputable sources like CNN, BBC, or the official White House website. They should have the most recent updates on the current President of the United States.

Would you like to know more about Joe Biden or the US presidency in general?

Assistant: You're probably thinking of Hunter Biden. He's the eldest son of President Joe Biden and his late wife, Neilia Biden. Hunter Biden has been in the public eye due to his involvement in various business ventures and his personal life.

Hunter Biden has been involved in several high-profile controversies, including his work with Ukrainian energy company Burisma Holdings and his struggles wit

The `chat_repl()` method stands for **R**ead-**E**val-**P**rint **L**oop. By using it, your code becomes cleaner and less error-prone, as you don't have to manage the state of the conversation loop yourself.

-----

## 5.&nbsp; Run the Final Application

With the final `chat_repl()` version in place, run the application from the VSCode terminal to start your conversation.

In [ ]:
# python main.py

To exit the conversation, simply type `exit`.

-----

## 6.&nbsp; Challenges and Further Exploration 😀

Now that you have a conversational chatbot, it's time to experiment.

### 1. Master the System Prompt

The `LLM_SYSTEM_PROMPT` in `src/config.py` is your primary tool for controlling the chatbot's personality.

**Challenge:**

  * **Change the Persona:** Modify the system prompt to make the chatbot act like a grumpy pirate, a cheerful robot, or a world-weary detective. How does the tone and phrasing of its answers change?
  * **Add Constraints:** Add a rule to the prompt, such as "You must always answer in the form of a haiku" or "Never use the letter 'e' in your responses." See how well the model adheres to these constraints.

### 2. Solve the Context Window Problem

As we discussed, `SimpleChatEngine` can be inefficient for long conversations. A better approach is to use an engine that summarises the conversation as it goes.

**Your Next Step: Implementing `CondenseQuestionChatEngine`**

This engine works differently: on each turn, it first "condenses" your new question with the chat history into a single, more focused question for the LLM. This is a great way to maintain context without sending an ever-growing history.

**How to Implement It:**

1.  Open `src/engine/engine.py`.
2.  Change your import from `SimpleChatEngine` to `CondenseQuestionChatEngine`.
3.  In `main_chat_loop`, change the class used to create the `conversation` object to `CondenseQuestionChatEngine`.

**Challenge:** Run the application with this new engine. Does the conversation feel different? Now that you know the pattern for changing the engine, try exploring other memory types from the LlamaIndex documentation and applying them.

### 3. Combine Everything

The true power of this platform lies in combining different models, prompts, and memory types.

**Challenge:** Experiment with various combinations. Which setup works best for you and why? Consider the trade-offs between response quality, speed, and conversational coherence. Try using a different model (from the first guide) with the `CondenseQuestionChatEngine` and a various prompts. Note your results, then experiment again and make more notes.